In [ ]:
# --- install first (then import) ---
!pip -q install -U "tensorflow==2.17.*" "tensorflow-datasets>=4.9,<5"

# If you previously installed a different TF version in this session, uncomment the next line to restart:
# import os, sys; os.kill(os.getpid(), 9)

import tensorflow as tf
import pandas as pd
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)


In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
train_df = pd.read_csv(train_file_path, sep='\t', names=['label', 'message'])
test_df = pd.read_csv(test_file_path, sep='\t', names=['label', 'message'])

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")


In [ ]:
train_df.head()

In [ ]:
train_df[train_df['label'] == 'spam'].head()


In [ ]:
test_df.head()

In [ ]:
train_df.info()

In [ ]:
test_df.info()

In [ ]:
# Step 1 — encode labels and split features/targets
label_map = {'ham': 0, 'spam': 1}

y_train = train_df['label'].map(label_map).astype('int32').values
y_test  = test_df['label'].map(label_map).astype('int32').values

X_train_text = train_df['message'].astype(str).values
X_test_text  = test_df['message'].astype(str).values

# quick sanity checks
print("Train label counts [ham, spam]:", np.bincount(y_train))
print("Example:", X_train_text[0], "->", y_train[0])


In [ ]:
from tensorflow.keras.layers import TextVectorization

# 1) Set simple, reasonable limits
max_tokens = 20000      # keep only the top 20k most frequent words
seq_len    = 100        # each message will be padded/truncated to 100 tokens

# 2) Create the vectorizer: how to clean + tokenize + output
vectorizer = TextVectorization(
    max_tokens=max_tokens,
    standardize='lower_and_strip_punctuation',  # lowercase + remove punctuation
    split='whitespace',                         # split on spaces
    output_mode='int',                          # map words -> integer ids
    output_sequence_length=seq_len              # pad/truncate to fixed length
)

# 3) "Learn" the vocabulary from the training text only
text_ds = tf.data.Dataset.from_tensor_slices(X_train_text).batch(64)
vectorizer.adapt(text_ds)

# 4) Vectorize (integerize) the texts
X_train = vectorizer(X_train_text)   # shape: (num_examples, seq_len), dtype int
X_test  = vectorizer(X_test_text)



# 6) Quick sanity checks
vocab = vectorizer.get_vocabulary()
print("Vocab size (including OOV):", len(vocab))
print("First vocab entries:", vocab[:10])
print("One vectorized sample (first 20 ids):", X_train[0][:20].numpy())


In [ ]:
# 1) Basic sizes
vocab_size    = len(vectorizer.get_vocabulary())  # how many tokens our vectorizer knows
embedding_dim = 64                                 # size of word embeddings
rnn_units     = 64                                 # LSTM hidden size
batch_size    = 64
epochs        = 10

# 2) Model: Embedding -> LSTM -> Dense(1, sigmoid)
model = keras.Sequential([
    keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=seq_len,
        mask_zero=True  # ignore padding (0) so LSTM focuses on real tokens
    ),
    keras.layers.LSTM(rnn_units),
    keras.layers.Dense(1, activation="sigmoid")  # binary classification: spam(1)/ham(0)
])

# 3) Quick look at the model
model.summary()

In [ ]:
# 4) Compile: loss for binary labels, a simple optimizer, and accuracy metric
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [ ]:
# 5) Train: use a small validation split from the training set
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=epochs,
    batch_size=batch_size,
    verbose=1
)

In [ ]:
# 6) Evaluate on the held-out test set
test_loss, test_acc = model.evaluate(X_test, y_test, batch_size=batch_size, verbose=0)
print(f"Test loss: {test_loss:.4f}  |  Test accuracy: {test_acc:.4f}")

In [ ]:
# function to predict messages based on model
# (returns [probability, label])
def predict_message(pred_text):
  # 1) Convert text → integer sequence using the same vectorizer
  vectorized_text = vectorizer([pred_text])   # shape: (1, seq_len)

  # 2) Get model prediction (a number between 0 and 1)
  pred = model.predict(vectorized_text)[0][0]

  # 3) Decide label: <0.5 → ham, >=0.5 → spam
  label = "spam" if pred >= 0.5 else "ham"

  # 4) Return as list [probability, label]
  return [float(pred), label]

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()


# Source:

This is my Free Code Camp training project solution:
https://www.freecodecamp.org/learn/machine-learning-with-python/machine-learning-with-python-projects/neural-network-sms-text-classifier